# Wide World Importers - Business Queries
This notebook connects to the `asb_olap` dimensional database to answer the 10 business questions from the case study.

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
db_user = os.getenv('DB_USER', 'db_user')
db_password = os.getenv('DB_PASSWORD', 'db_password')
db_host = os.getenv('DB_HOST', 'localhost')
db_port = os.getenv('DB_PORT', '5432')
db_name = os.getenv('DB_NAME_OLAP', 'asb_olap')

engine = create_engine(f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}')

def run_query(sql):
    return pd.read_sql(sql, engine)

### Q1. What are total sales, quantity sold, and profit by month and fiscal year?

In [2]:
q1 = """
SELECT 
    d.fiscal_year,
    d.month_name,
    SUM(f.extended_price) AS total_sales,
    SUM(f.invoiced_quantity) AS total_quantity_sold,
    SUM(f.line_profit) AS total_profit
FROM fact_sale f
JOIN dim_date d ON f.invoice_date_key = d.date_key
GROUP BY d.fiscal_year, d.month_number, d.month_name
ORDER BY d.fiscal_year, d.month_number;
"""
run_query(q1).head(12)

,fiscal_year,month_name,total_sales,total_quantity_sold,total_profit
0,2013,January,4335972.97,193271,1890687.80
1,2013,February,3193304.60,142120,1383023.30
2,2013,March,4451081.62,207486,1933679.05
3,2013,April,4668548.40,212995,2011957.20
4,2013,May,5080661.05,230725,2231604.45
5,2013,June,4679392.14,213468,2003843.25
6,2013,July,5039033.18,232599,2177622.75
7,2013,August,4020390.18,192199,1735165.70
8,2013,September,4345897.56,190567,1893782.75
9,2013,October,4315500.25,198476,1882507.65


### Q2. Which products and product categories generate the highest sales and profit?

In [3]:
q2 = """
SELECT 
    s.stock_item_name,
    s.package_type_name,
    SUM(f.extended_price) AS total_sales,
    SUM(f.line_profit) AS total_profit
FROM fact_sale f
JOIN dim_stock_item s ON f.stock_item_key = s.stock_item_key
GROUP BY s.stock_item_name, s.package_type_name
ORDER BY total_profit DESC
LIMIT 10;
"""
run_query(q2)

,stock_item_name,package_type_name,total_sales,total_profit
0,20 mm Double sided bubble wrap 50m,Each,7146468.00,5293680.0
1,Air cushion machine (Blue),Each,12773338.65,4439391.0
2,32 mm Anti static bubble wrap (Blue) 50m,Each,7341600.00,3526400.0
3,10 mm Anti static bubble wrap (Blue) 50m,Each,7278430.50,3452220.0
4,32 mm Double sided bubble wrap 50m,Each,7118776.00,2929310.0
5,10 mm Double sided bubble wrap 50m,Each,6834450.00,2773400.0
6,20 mm Anti static bubble wrap (Blue) 50m,Each,6664986.00,2670540.0
7,32 mm Anti static bubble wrap (Blue) 20m,Each,3335184.00,1510500.0
8,Void fill 400 L bag (White) 400L,Each,3302225.00,1378320.0
9,32 mm Double sided bubble wrap 20m,Each,2447901.50,1323190.0


### Q3. Which customers and customer categories contribute the most revenue?

In [4]:
q3 = """
SELECT 
    c.customer_name,
    c.customer_category_name,
    SUM(f.extended_price) AS total_revenue
FROM fact_sale f
JOIN dim_customer c ON f.customer_key = c.customer_key
GROUP BY c.customer_name, c.customer_category_name
ORDER BY total_revenue DESC
LIMIT 10;
"""
run_query(q3)

,customer_name,customer_category_name,total_revenue
0,"Tailspin Toys (Inguadona, MN)",Novelty Shop,438689.81
1,"Tailspin Toys (Minidoka, ID)",Novelty Shop,427445.57
2,Mauno Laurila,Novelty Shop,424324.76
3,"Wingtip Toys (Sarversville, PA)",Novelty Shop,420001.08
4,Nasrin Omidzadeh,Supermarket,416005.03
5,"Tailspin Toys (Long Meadow, MD)",Novelty Shop,414934.18
6,Ingrida Zeltina,Supermarket,413817.88
7,"Wingtip Toys (San Jacinto, CA)",Novelty Shop,408524.95
8,Laszlo Gardenier,Gift Store,407800.89
9,Camille Authier,Gift Store,406493.27


### Q4. How do sales and profit vary by city, state or province, and sales territory?

In [5]:
q4 = """
SELECT 
    ci.city_name,
    ci.state_province_name,
    ci.region AS sales_territory,
    SUM(f.extended_price) AS total_sales,
    SUM(f.line_profit) AS total_profit
FROM fact_sale f
JOIN dim_customer c ON f.customer_key = c.customer_key
-- We can join ci through the customer's delivery city name, or join dim_city directly if we had a city_key in the fact. 
-- Since we embedded delivery_city_name in dim_customer, we can use that, or join dim_city to get state and region:
JOIN dim_city ci ON c.delivery_city_name = ci.city_name
GROUP BY ci.city_name, ci.state_province_name, ci.region
ORDER BY total_sales DESC
LIMIT 10;
"""
run_query(q4)

,city_name,state_province_name,sales_territory,total_sales,total_profit
0,Sinclair,Wyoming,Americas,648919.36,288943.40
1,Teutopolis,Illinois,Americas,639424.77,268520.65
2,East Fultonham,Ohio,Americas,628838.61,280544.80
3,Akhiok,Alaska,Americas,623650.80,259554.30
4,Cherry Grove Beach,South Carolina,Americas,605388.69,257902.75
5,Rockwall,Texas,Americas,582839.74,253225.35
6,Inguadona,Minnesota,Americas,438689.81,189277.45
7,Minidoka,Idaho,Americas,427445.57,185432.55
8,North Eaton,Ohio,Americas,424324.76,172559.80
9,Sarversville,Pennsylvania,Americas,420001.08,188919.75


### Q5. Which employees or salespeople manage the highest-value orders and sales?

In [6]:
q5 = """
SELECT 
    e.full_name AS salesperson_name,
    COUNT(DISTINCT f.invoice_id) AS total_invoices,
    SUM(f.extended_price) AS total_sales_value
FROM fact_sale f
JOIN dim_employee e ON f.salesperson_key = e.employee_key
GROUP BY e.full_name
ORDER BY total_sales_value DESC;
"""
run_query(q5)

,salesperson_name,total_invoices,total_sales_value
0,Archer Lamble,7184,20629803.44
1,Kayla Woodcock,7160,20197271.36
2,Hudson Onslow,6988,19906785.93
3,Taj Shand,7074,19882524.15
4,Sophia Hinton,7073,19866023.48
5,Hudson Hollinworth,7079,19697857.73
6,Lily Code,6978,19650933.07
7,Jack Potter,7061,19628351.36
8,Amy Trefl,6959,19293258.55
9,Anthony Grosse,6954,19290630.38


### Q6. What proportion of ordered quantities is later invoiced?

In [7]:
q6 = """
-- This query compares total ordered quantity vs total invoiced quantity across the entire dataset.
SELECT 
    (SELECT SUM(ordered_quantity) FROM fact_order) AS total_ordered,
    (SELECT SUM(invoiced_quantity) FROM fact_sale) AS total_invoiced,
    ROUND(CAST((SELECT SUM(invoiced_quantity) FROM fact_sale) AS DECIMAL) / 
          CAST((SELECT SUM(ordered_quantity) FROM fact_order) AS DECIMAL) * 100, 2) AS proportion_percent;
"""
run_query(q6)

,total_ordered,total_invoiced,proportion_percent
0,9310904,8950628,96.13


### Q7. How long does it take, on average, for an order to be invoiced?

In [8]:
q7 = """
SELECT 
    ROUND(AVG(days_from_order_to_invoice), 2) AS avg_days_to_invoice
FROM fact_sale;
"""
run_query(q7)

,avg_days_to_invoice
0,1.33


### Q8. Which orders contain backordered items, and what is their business impact?

In [9]:
q8 = """
SELECT 
    COUNT(DISTINCT order_id) AS total_backordered_orders,
    SUM(ordered_quantity - COALESCE(picked_quantity, 0)) AS total_items_delayed,
    SUM((ordered_quantity - COALESCE(picked_quantity, 0)) * unit_price) AS estimated_revenue_delayed
FROM fact_order
WHERE is_undersupply_backordered = TRUE;
"""
run_query(q8)

,total_backordered_orders,total_items_delayed,estimated_revenue_delayed
0,73595,360276,5372935.2


### Q9 & Q10. Slowly Changing Dimensions (Tracking Historical Changes)

In [10]:
q9_10 = """
-- To demonstrate this, let's look at a customer that might have multiple records if we ran an update.
-- Since we only loaded a single snapshot, most customers will have 1 active record.
SELECT 
    customer_id, 
    customer_name, 
    customer_category_name, 
    valid_from, 
    valid_to, 
    is_current 
FROM dim_customer 
ORDER BY customer_id LIMIT 5;
"""
run_query(q9_10)

,customer_id,customer_name,customer_category_name,valid_from,valid_to,is_current
0,1,Tailspin Toys (Head Office),Novelty Shop,2026-10-07 13:17:21.750947,9999-12-31 23:59:59,True
1,2,"Tailspin Toys (Sylvanite, MT)",Novelty Shop,2026-10-07 13:17:21.750947,9999-12-31 23:59:59,True
2,3,"Tailspin Toys (Peeples Valley, AZ)",Novelty Shop,2026-10-07 13:17:21.750947,9999-12-31 23:59:59,True
3,4,"Tailspin Toys (Medicine Lodge, KS)",Novelty Shop,2026-10-07 13:17:21.750947,9999-12-31 23:59:59,True
4,5,"Tailspin Toys (Gasport, NY)",Novelty Shop,2026-10-07 13:17:21.750947,9999-12-31 23:59:59,True
